# 09: Lock v3 (light) and one-shot evaluation on the fresh set

**Phase 1 (cells 1-4): validation only.** Writes `locked_config_v3.json`. Two confirmatory
claims, everything else explicitly exploratory with no verdict language.

**Phase 2 (cells 5-9): the fresh set.** Cell 5 refuses to load fresh predictions unless the
v3 lock is already saved in Drive. Confirmatory read first, then the exploratory analyses in
the same pass.

### Confirmatory (can fail)
1. **Primary.** Net severe misses (grade 3+) avoided at 25% matched referral, ensemble versus
   its better constituent, averaged over 55 pairs x 9 seed combinations. Patient-clustered
   bootstrap; supported if the 95% interval excludes zero.
2. **Capacity crossing.** Under enrichment to APTOS-like prevalence (40.6%, by dropping
   non-referable patients, all severe eyes retained), the same advantage is predicted to be
   **negative at 25% burden** (below prevalence) and **positive at 50%** (above it), with the
   unenriched set at 25% as positive control.

### Exploratory (reported with intervals, no pass/fail)
Number needed to refer at natural operating points, with the severe-miss reduction split into
a volume effect and a complementarity effect; validation rho as an out-of-sample predictor of
fresh advantage; the severity-weighted cost index across the 25-cell weight grid; the witness
and engineering pairs carried over from lock v2; the 10-50% curves; grade-4-only misses.

Benchmark for interpretation, not a gate: one additional severe case caught per 1,000
patients screened at identical referral volume.

CPU runtime. About 30-40 minutes, mostly the bootstrap. Run cells 1-9 in order.

In [1]:
# Cell 1: auth; load VALIDATION predictions only; carry over lock v2
import io, os, json, hashlib, datetime, itertools, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_RESULTS     = '1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'
F_FIGURES     = '18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done = False
        while not done: _, done = d.next_chunk()
def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def up(path, folder, mime):
    svc.files().create(body={'name': os.path.basename(path), 'parents': [folder]},
                       media_body=MediaFileUpload(path, mimetype=mime), fields='id').execute()
    print('  uploaded', os.path.basename(path))
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

if find_id('locked_config_v3.json', F_RESULTS):
    raise SystemExit('locked_config_v3.json exists. Do not re-lock.')
v2id = find_id('locked_config.json', F_RESULTS); dl(v2id, 'lock_v2.json')
LOCK2 = json.load(open('lock_v2.json')); LOCK2_SHA = sha256('lock_v2.json')
WIT  = (LOCK2['tier2_witness']['model_a'], LOCK2['tier2_witness']['model_b'])
ENG  = (LOCK2['tier3_engineering']['model_a'], LOCK2['tier3_engineering']['model_b'])
CTRL = ('densenet121_binary', 'retfound_binary')
RHO_VAL = {tuple(k.split('|')): v for k, v in LOCK2['rho']['validation'].items()}

ARCHS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
         'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
         'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = [0, 1, 2]; PAIRS = list(itertools.combinations(ARCHS, 2))
def tag(k): return '' if k == 0 else f'_seed{k}'

PV, HV, refv = {}, {}, None
for a in ARCHS:
    for k in SEEDS:
        fn = f'eyepacs_predictions_val_{a}{tag(k)}_v2.csv'
        dl(find_id(fn, F_PREDICTIONS), fn); d = pd.read_csv(fn); HV[fn] = sha256(fn)
        assert set(d['split']) == {'val'}
        if refv is None: refv = d[['image_id','true_grade','true_label']].reset_index(drop=True)
        PV[(a, k)] = d['predicted_probability'].to_numpy(np.float64)
LABV, GRDV = refv['true_label'].to_numpy(int), refv['true_grade'].to_numpy(int)
print(f'validation loaded: {len(LABV)} images, severe {(GRDV>=3).sum()} | witness {WIT} | engineering {ENG}')

validation loaded: 2900 images, severe 132 | witness ('vit_base_5class', 'retfound_adversarial') | engineering ('densenet121_binary', 'retfound_adversarial')


In [2]:
# Cell 2: locked definitions
BURDENS = [round(x, 3) for x in np.arange(0.10, 0.5001, 0.025)]
PRIMARY = 0.25; CAP_LOW = 0.25; CAP_HIGH = 0.50
W = {'w4': 500, 'w3': 200, 'w2': 15, 'wfp': 1}                 # secondary only
GRID = {'miss': [[500,200,15],[250,100,10],[1000,400,20],[100,50,10],[50,25,5]], 'wfp': [1,5,20,50,100]}
ENRICH_TARGET = 0.406; ENRICH_DRAWS = 50

def sev_curve(score, sev, Rs):
    """Severe cases NOT referred when referring the top-R by score. Selection is label-free."""
    o = np.argsort(-score, kind='stable'); cs = np.cumsum(sev[o])
    return sev.sum() - cs[Rs - 1]

def cost_curve(score, missw, neg, Rs, wfp):
    o = np.argsort(-score, kind='stable')
    return (missw.sum() - np.cumsum(missw[o])[Rs - 1]) + wfp * np.cumsum(neg[o])[Rs - 1]

def prep(idx, LAB, GRD, burdens=BURDENS, severe_min=3):
    lab, grd = LAB[idx], GRD[idx]; n = len(idx)
    return {'n': n, 'lab': lab, 'grd': grd,
            'sev': (grd >= severe_min).astype(float),
            'Rs': np.array([int(round(r * n)) for r in burdens])}

def pair_advantage(idx, PD, C):
    """Net severe misses avoided vs the better constituent, per pair (mean over 9 seed combos).
    Rank-based OR-gate; both constituents thresholded to the same referral count (label-free).
    'Better constituent' is the one with fewer severe misses on the evaluation set: conservative."""
    sub = {key: p[idx] for key, p in PD.items()}
    rk = {key: rankdata(v) / C['n'] for key, v in sub.items()}
    single = {key: sev_curve(v, C['sev'], C['Rs']) for key, v in sub.items()}
    out = {}
    for a, b in PAIRS:
        acc = []
        for i in SEEDS:
            for j in SEEDS:
                orm = sev_curve(np.maximum(rk[(a, i)], rk[(b, j)]), C['sev'], C['Rs'])
                acc.append(np.minimum(single[(a, i)], single[(b, j)]) - orm)
        out[(a, b)] = np.array(acc)                      # 9 x burdens
    return out, single

def mean_adv(adv):
    return np.vstack([adv[p].mean(0) for p in PAIRS]).mean(0)
print('definitions set | burdens', BURDENS)

definitions set | burdens [np.float64(0.1), np.float64(0.125), np.float64(0.15), np.float64(0.175), np.float64(0.2), np.float64(0.225), np.float64(0.25), np.float64(0.275), np.float64(0.3), np.float64(0.325), np.float64(0.35), np.float64(0.375), np.float64(0.4), np.float64(0.425), np.float64(0.45), np.float64(0.475), np.float64(0.5)]


In [3]:
# Cell 3: validation values (context for the lock; no decision rests on these)
CV = prep(np.arange(len(LABV)), LABV, GRDV)
ADV_V, SINGLE_V = pair_advantage(np.arange(len(LABV)), PV, CV)
mv = mean_adv(ADV_V); t25 = BURDENS.index(PRIMARY)
print(f'validation, 25% burden: mean net severe misses avoided per pair {mv[t25]:.2f} '
      f'| pairs positive {int((np.vstack([ADV_V[p].mean(0) for p in PAIRS])[:, t25] > 0).sum())}/55')
print('validation severe cases:', int(CV['sev'].sum()))

validation, 25% burden: mean net severe misses avoided per pair 0.85 | pairs positive 41/55
validation severe cases: 132


In [4]:
# Cell 4: write lock v3
LOCK3 = {
 'locked_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'locked_on': 'EyePACS validation predictions only (33 files); no fresh prediction opened',
 'validation_file_sha256': HV, 'lock_v2_sha256': LOCK2_SHA,
 'carried_from_v2': {'witness': WIT, 'engineering': ENG, 'control_added_pre_test': CTRL,
                     'rho_validation': 'reused from lock v2 (same definition, same validation data)'},
 'or_gate': 'rank-based: max of within-set ranks; both models at the same individual referral rate',
 'referral_matching': 'refer top round(r*n) by score; label-free',
 'confirmatory': {
   'primary': {'statistic': 'net severe misses (grade>=3) avoided vs better constituent at 25% burden, '
                            'mean over 55 pairs x 9 seed combinations',
               'uncertainty': 'patient-clustered bootstrap, 1000 resamples, patients resampled jointly',
               'supported_if': '95% interval excludes zero',
               'note': 'model-level resampling reported alongside but is not the decision rule'},
   'capacity_crossing': {'enrichment': f'drop non-referable patients at random until referable rate '
                                       f'= {ENRICH_TARGET}; all severe eyes retained; {ENRICH_DRAWS} draws',
                         'prediction': 'mean advantage negative at 25% burden and positive at 50%',
                         'positive_control': 'unenriched fresh set at 25%',
                         'limit': 'shows prevalence is sufficient to flip the sign; does not prove it caused '
                                  'the APTOS reversal (APTOS also differs in grading, cameras, population)'}},
 'exploratory_no_verdict': ['number needed to refer at natural operating points, split into volume and '
                            'complementarity effects', 'validation rho as out-of-sample predictor of fresh advantage',
                            'severity-weighted cost index across the 25-cell weight grid',
                            'witness and engineering pairs on the fresh set', '10-50% curves',
                            'grade-4-only misses'],
 'benchmark_not_a_gate': '1 additional severe case caught per 1,000 patients screened at equal referral volume',
 'cost_weights_secondary': W, 'weight_grid': GRID, 'burdens': BURDENS,
 'validation_context': {'mean_net_severe_avoided_at_25pct': round(float(mv[t25]), 3),
                        'validation_severe_cases': int(CV['sev'].sum())},
}
with open('locked_config_v3.json', 'w') as f: json.dump(LOCK3, f, indent=2, default=str)
up('locked_config_v3.json', F_RESULTS, 'application/json')
LOCK3_SHA = sha256('locked_config_v3.json')
print('LOCK v3 WRITTEN', LOCK3['locked_at_utc'], '| sha256', LOCK3_SHA[:16])

  uploaded locked_config_v3.json
LOCK v3 WRITTEN 2026-09-22T21:39:22.157905+00:00 | sha256 9cf7292181c56165


In [5]:
# Cell 5: PHASE 2 GATE: only now load the fresh predictions
assert find_id('locked_config_v3.json', F_RESULTS), 'lock v3 must be saved before opening the fresh set'
PF, HF, reff = {}, {}, None
for a in ARCHS:
    for k in SEEDS:
        fn = f'eyepacs_predictions_fresh_{a}{tag(k)}_v2.csv'
        dl(find_id(fn, F_PREDICTIONS), fn); d = pd.read_csv(fn); HF[fn] = sha256(fn)
        assert set(d['split']) == {'fresh'}
        if reff is None: reff = d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
        assert (d['image_id'].astype(str).values == reff['image_id'].astype(str).values).all()
        PF[(a, k)] = d['predicted_probability'].to_numpy(np.float64)
LABF = reff['true_label'].to_numpy(int); GRDF = reff['true_grade'].to_numpy(int)
PIDF = reff['patient_id'].astype(str).to_numpy()
N = len(LABF); PATS = pd.unique(PIDF); PAT_IMGS = [np.where(PIDF == p)[0] for p in PATS]
print(f'fresh: {N} images, {len(PATS)} patients, referable {LABF.mean():.4f}, '
      f'severe {(GRDF>=3).sum()}, grade4 {(GRDF==4).sum()}')

fresh: 15794 images, 7897 patients, referable 0.1953, severe 712, grade4 328


In [6]:
# Cell 6: CONFIRMATORY 1: primary, with patient-clustered bootstrap
ALL = np.arange(N); CF = prep(ALL, LABF, GRDF)
ADV_F, SINGLE_F = pair_advantage(ALL, PF, CF)
pair_mean = np.vstack([ADV_F[p].mean(0) for p in PAIRS])        # 55 x burdens
point = pair_mean.mean(0)

rng = np.random.default_rng(20260922); B = 1000; boot = []
t0 = time.time()
for b in range(B):
    idx = np.concatenate([PAT_IMGS[i] for i in rng.integers(0, len(PATS), len(PATS))])
    C = prep(idx, LABF, GRDF)
    boot.append(mean_adv(pair_advantage(idx, PF, C)[0]))
    if (b + 1) % 100 == 0: print(f'  bootstrap {b+1}/{B} ({time.time()-t0:.0f}s)')
boot = np.array(boot); lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

# model-level resampling (reported, not the decision rule)
ml = []
for _ in range(2000):
    samp = rng.choice(ARCHS, len(ARCHS), replace=True)
    ps = [(x, y) if (x, y) in ADV_F else (y, x) for x, y in itertools.combinations(samp, 2) if x != y]
    if len(ps) < 5: continue
    ml.append(np.mean([ADV_F[p].mean(0)[t25] for p in ps]))
ml_lo, ml_hi = np.percentile(ml, [2.5, 97.5])

per1000 = point[t25] / len(PATS) * 1000
PRIM = {'mean_net_severe_avoided_per_pair': round(float(point[t25]), 3),
        'ci95': [round(float(lo[t25]), 3), round(float(hi[t25]), 3)],
        'supported': bool(lo[t25] > 0),
        'pairs_positive_of_55': int((pair_mean[:, t25] > 0).sum()),
        'model_level_ci95_reported_only': [round(float(ml_lo), 3), round(float(ml_hi), 3)],
        'per_1000_patients_screened': round(float(per1000), 2),
        'benchmark_per_1000': 1.0}
print('\nCONFIRMATORY 1 (primary):', json.dumps(PRIM, indent=2))

  bootstrap 100/1000 (107s)
  bootstrap 200/1000 (212s)
  bootstrap 300/1000 (317s)
  bootstrap 400/1000 (421s)
  bootstrap 500/1000 (526s)
  bootstrap 600/1000 (631s)
  bootstrap 700/1000 (736s)
  bootstrap 800/1000 (840s)
  bootstrap 900/1000 (945s)
  bootstrap 1000/1000 (1050s)

CONFIRMATORY 1 (primary): {
  "mean_net_severe_avoided_per_pair": 4.741,
  "ci95": [
    2.31,
    6.15
  ],
  "supported": true,
  "pairs_positive_of_55": 48,
  "model_level_ci95_reported_only": [
    2.513,
    7.51
  ],
  "per_1000_patients_screened": 0.6,
  "benchmark_per_1000": 1.0
}


In [7]:
# Cell 7: CONFIRMATORY 2: capacity crossing under enrichment
pat_ref = pd.DataFrame({'p': PIDF, 'ref': LABF}).groupby('p')['ref'].max()
clean_pats = pat_ref[pat_ref == 0].index.to_numpy()            # patients with no referable eye
n_ref_img = int(LABF.sum())
target_total = int(round(n_ref_img / ENRICH_TARGET))
drop_imgs = N - target_total; drop_pats = drop_imgs // 2
print(f'enrichment: keep all {n_ref_img} referable images, drop {drop_pats} of {len(clean_pats)} '
      f'non-referable patients -> ~{target_total} images at ~{ENRICH_TARGET:.3f} referable')

cap_rows = []
rng2 = np.random.default_rng(20260925)
for d in range(ENRICH_DRAWS):
    drop = set(rng2.choice(clean_pats, drop_pats, replace=False))
    keep = np.array([i for i in range(N) if PIDF[i] not in drop])
    C = prep(keep, LABF, GRDF, burdens=[CAP_LOW, CAP_HIGH])
    adv, _ = pair_advantage(keep, PF, C)
    m = np.vstack([adv[p].mean(0) for p in PAIRS]).mean(0)
    cap_rows.append({'draw': d, 'referable_rate': float(LABF[keep].mean()),
                     'adv_at_25': float(m[0]), 'adv_at_50': float(m[1])})
CAP = pd.DataFrame(cap_rows)
CAPRES = {'enriched_referable_rate': round(float(CAP['referable_rate'].mean()), 4),
          'adv_at_25_mean': round(float(CAP['adv_at_25'].mean()), 3),
          'adv_at_25_pct_range': [round(float(x), 3) for x in np.percentile(CAP['adv_at_25'], [2.5, 97.5])],
          'adv_at_50_mean': round(float(CAP['adv_at_50'].mean()), 3),
          'adv_at_50_pct_range': [round(float(x), 3) for x in np.percentile(CAP['adv_at_50'], [2.5, 97.5])],
          'positive_control_unenriched_at_25': round(float(point[t25]), 3),
          'crossing_as_predicted': bool(CAP['adv_at_25'].mean() < 0 < CAP['adv_at_50'].mean())}
print('\nCONFIRMATORY 2 (capacity crossing):', json.dumps(CAPRES, indent=2))

enrichment: keep all 3085 referable images, drop 4097 of 6109 non-referable patients -> ~7599 images at ~0.406 referable

CONFIRMATORY 2 (capacity crossing): {
  "enriched_referable_rate": 0.4059,
  "adv_at_25_mean": 4.36,
  "adv_at_25_pct_range": [
    3.884,
    4.864
  ],
  "adv_at_50_mean": 3.122,
  "adv_at_50_pct_range": [
    2.896,
    3.312
  ],
  "positive_control_unenriched_at_25": 4.741,
  "crossing_as_predicted": false
}


In [8]:
# Cell 8: EXPLORATORY (no verdicts)
# 8a. number needed to refer at natural operating points, volume vs complementarity
nv = len(LABV); Rv = int(round(PRIMARY * nv))
thr = {key: np.sort(p)[::-1][Rv - 1] for key, p in PV.items()}      # each model's own 25% threshold on validation
sevF = (GRDF >= 3).astype(float)
def refset(mask): return mask.astype(int)
best_single = ARCHS[int(np.argmin([np.mean([SINGLE_F[(a, k)][t25] for k in SEEDS]) for a in ARCHS]))]
rows = []
for a, b in PAIRS:
    for k in SEEDS:
        ma = PF[(a, k)] >= thr[(a, k)]; mb = PF[(b, k)] >= thr[(b, k)]
        orm = ma | mb
        base = ma if np.sum(sevF[~ma]) <= np.sum(sevF[~mb]) else mb
        R_or = int(orm.sum())
        base_at_Ror = np.zeros(N, bool)
        p_base = PF[(a, k)] if base is ma else PF[(b, k)]
        base_at_Ror[np.argsort(-p_base, kind='stable')[:R_or]] = True
        rows.append({'pair': f'{a}|{b}', 'seed': k,
                     'extra_referrals': R_or - int(base.sum()),
                     'extra_false_positives': int(((orm) & (LABF == 0)).sum() - ((base) & (LABF == 0)).sum()),
                     'severe_avoided_total': float(np.sum(sevF[~base]) - np.sum(sevF[~orm])),
                     'severe_avoided_volume': float(np.sum(sevF[~base]) - np.sum(sevF[~base_at_Ror])),
                     'severe_avoided_complementarity': float(np.sum(sevF[~base_at_Ror]) - np.sum(sevF[~orm]))})
NNR = pd.DataFrame(rows)
tot = NNR['severe_avoided_total'].sum()
NNRES = {'best_single_at_25pct': best_single,
         'median_extra_referrals': float(NNR['extra_referrals'].median()),
         'median_extra_false_positives': float(NNR['extra_false_positives'].median()),
         'mean_severe_avoided': round(float(NNR['severe_avoided_total'].mean()), 2),
         'number_needed_to_refer': round(float(NNR['extra_referrals'].sum() / tot), 1) if tot else None,
         'share_volume_effect': round(float(NNR['severe_avoided_volume'].sum() / tot), 3) if tot else None,
         'share_complementarity': round(float(NNR['severe_avoided_complementarity'].sum() / tot), 3) if tot else None}
print('8a NNR / decomposition:', json.dumps(NNRES, indent=2))

# 8b. validation rho as out-of-sample predictor of fresh advantage
adv25 = {p: float(ADV_F[p].mean(0)[t25]) for p in PAIRS}
xs = [RHO_VAL[p] for p in PAIRS]
rs = spearmanr(xs, [adv25[p] for p in PAIRS]).correlation
mlr = []
for _ in range(2000):
    samp = rng.choice(ARCHS, len(ARCHS), replace=True)
    ps = [(x, y) if (x, y) in adv25 else (y, x) for x, y in itertools.combinations(samp, 2) if x != y]
    if len(set(ps)) < 5: continue
    mlr.append(spearmanr([RHO_VAL[p] for p in ps], [adv25[p] for p in ps]).correlation)
RHORES = {'spearman_validation_rho_vs_fresh_advantage': round(float(rs), 3),
          'model_level_ci95': [round(float(x), 3) for x in np.nanpercentile(mlr, [2.5, 97.5])],
          'direction_predicted': 'negative'}
print('8b rho out-of-sample:', json.dumps(RHORES, indent=2))

# 8c. cost index across the weight grid at 25%
grid_rows = []
for w4, w3, w2 in GRID['miss']:
    missw = np.where(LABF == 1, np.where(GRDF == 4, w4, np.where(GRDF == 3, w3, w2)), 0).astype(float)
    neg = (LABF == 0).astype(float)
    for wfp in GRID['wfp']:
        rk = {key: rankdata(p) / N for key, p in PF.items()}
        sc = {key: cost_curve(p, missw, neg, CF['Rs'], wfp) for key, p in PF.items()}
        vals = []
        for a, b in PAIRS:
            for i in SEEDS:
                for j in SEEDS:
                    c_or = cost_curve(np.maximum(rk[(a, i)], rk[(b, j)]), missw, neg, CF['Rs'], wfp)
                    vals.append(np.minimum(sc[(a, i)], sc[(b, j)])[t25] - c_or[t25])
        grid_rows.append({'w4/w3/w2': f'{w4}/{w3}/{w2}', 'wfp': wfp, 'mean_cost_advantage': round(float(np.mean(vals)), 1),
                          'pairs_positive': int(np.sum(np.array(vals).reshape(55, 9).mean(1) > 0))})
GRIDF = pd.DataFrame(grid_rows)
print('\n8c cost index across weight grid at 25%:')
print(GRIDF.pivot(index='w4/w3/w2', columns='wfp', values='mean_cost_advantage').to_string())

# 8d. witness / engineering / control on the fresh set
DIAG = [0, 4, 8]
EXPAIRS = {'witness': float(ADV_F[WIT].mean(0)[t25]),
           'engineering_matched_seeds': float(ADV_F[ENG][DIAG].mean(0)[t25]),
           'control_matched_seeds': float(ADV_F[CTRL][DIAG].mean(0)[t25])}
EXPAIRS['engineering_minus_control'] = round(EXPAIRS['engineering_matched_seeds'] - EXPAIRS['control_matched_seeds'], 3)
print('\n8d pairs carried over:', json.dumps({k: round(v, 3) for k, v in EXPAIRS.items()}, indent=2))

8a NNR / decomposition: {
  "best_single_at_25pct": "retfound_5class",
  "median_extra_referrals": 1278.0,
  "median_extra_false_positives": 1007.0,
  "mean_severe_avoided": 15.59,
  "number_needed_to_refer": 77.7,
  "share_volume_effect": 0.786,
  "share_complementarity": 0.214
}
8b rho out-of-sample: {
  "spearman_validation_rho_vs_fresh_advantage": 0.257,
  "model_level_ci95": [
    -0.413,
    0.679
  ],
  "direction_predicted": "negative"
}

8c cost index across weight grid at 25%:
wfp             1       5       20      50      100
w4/w3/w2                                           
100/50/10     870.3  1042.1  1645.1  2795.5  4700.5
1000/400/20  4804.5  5010.8  5762.1  7184.3  9401.3
250/100/10   1464.8  1655.2  2324.1  3573.6  5513.3
50/25/5       456.8   624.6  1206.6  2350.3  4251.9
500/200/15   2659.5  2856.8  3567.2  4890.2  6975.7

8d pairs carried over: {
  "witness": -0.222,
  "engineering_matched_seeds": 5.667,
  "control_matched_seeds": 2.333,
  "engineering_minus_cont

In [9]:
# Cell 9: curve figure and results file
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
x = np.array(BURDENS) * 100
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(x, point, 'o-', color='C0', label='fresh set (19.5% referable)')
ax.fill_between(x, lo, hi, color='C0', alpha=0.2)
ax.axhline(0, color='grey', lw=0.8); ax.axvline(PRIMARY * 100, ls=':', color='k')
ax.axvline(19.53, ls='--', color='C3', lw=1, label='prevalence')
ax.set_xlabel('Referral burden (%)'); ax.set_ylabel('Net severe misses avoided per pair')
ax.set_title('Ensemble advantage at matched referral burden (fresh set, 95% patient-clustered band)')
ax.legend(); fig.tight_layout(); fig.savefig('fresh_advantage_curve_v3.png', dpi=200)

CURVE = pd.DataFrame({'burden': BURDENS, 'mean_net_severe_avoided': point.round(3),
                      'lo': lo.round(3), 'hi': hi.round(3),
                      'pairs_positive_of_55': (pair_mean > 0).sum(0)})
RES = {'evaluated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'lock_v3_sha256': LOCK3_SHA, 'fresh_file_sha256': HF,
       'fresh_set': {'images': int(N), 'patients': int(len(PATS)), 'referable_rate': round(float(LABF.mean()), 4),
                     'severe': int((GRDF >= 3).sum()), 'grade4': int((GRDF == 4).sum())},
       'confirmatory_1_primary': PRIM, 'confirmatory_2_capacity': CAPRES,
       'exploratory': {'nnr_decomposition': NNRES, 'rho_out_of_sample': RHORES,
                       'weight_grid': grid_rows, 'carried_pairs': EXPAIRS},
       'curve': CURVE.to_dict(orient='records')}
with open('fresh_eval_results_v3.json', 'w') as f: json.dump(RES, f, indent=2, default=str)
CURVE.to_csv('fresh_curve_v3.csv', index=False)
NNR.to_csv('fresh_nnr_detail_v3.csv', index=False)
pd.DataFrame([{'model_a': p[0], 'model_b': p[1], 'rho_val': RHO_VAL[p], 'fresh_adv_25': round(adv25[p], 3)}
              for p in PAIRS]).to_csv('fresh_pairs_25pct_v3.csv', index=False)
for fn, m in (('fresh_eval_results_v3.json','application/json'), ('fresh_curve_v3.csv','text/csv'),
              ('fresh_nnr_detail_v3.csv','text/csv'), ('fresh_pairs_25pct_v3.csv','text/csv')):
    up(fn, F_RESULTS, m)
up('fresh_advantage_curve_v3.png', F_FIGURES, 'image/png')
print('\n' + CURVE.to_string(index=False))
print('\nPRIMARY:', PRIM['mean_net_severe_avoided_per_pair'], PRIM['ci95'], '->',
      'SUPPORTED' if PRIM['supported'] else 'not supported')
print('CAPACITY CROSSING:', 'as predicted' if CAPRES['crossing_as_predicted'] else 'not as predicted',
      f"(25%: {CAPRES['adv_at_25_mean']}, 50%: {CAPRES['adv_at_50_mean']})")

  uploaded fresh_eval_results_v3.json
  uploaded fresh_curve_v3.csv
  uploaded fresh_nnr_detail_v3.csv
  uploaded fresh_pairs_25pct_v3.csv
  uploaded fresh_advantage_curve_v3.png

 burden  mean_net_severe_avoided     lo    hi  pairs_positive_of_55
  0.100                    6.156  1.331 9.936                    40
  0.125                    6.519  1.991 9.684                    49
  0.150                    5.842  1.925 8.327                    51
  0.175                    4.265  0.711 6.544                    48
  0.200                    3.970  1.103 6.021                    39
  0.225                    4.806  2.061 6.507                    42
  0.250                    4.741  2.310 6.150                    48
  0.275                    4.176  1.725 5.925                    54
  0.300                    3.620  0.874 5.611                    53
  0.325                    3.725  0.781 5.312                    53
  0.350                    3.671  0.889 5.154                    54
  0.